[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-PY_Ed2026/blob/main/python/P06/P06-2_Etude_Ressort_CORRIGE.ipynb)

# Étude II : ressorts de compression

Étude de synthèse de la partie II du livre *Python pour l'ingénierie mécanique, industrielle et productique*.

## Cadrage

Cette étude réunit ce que les chapitres 4 à 6 ont enseigné : des tableaux NumPy et leurs tirages à graine fixée, les masques booléens, les histogrammes et les planches de matplotlib, la lecture en gris, les types et la mémoire, la mesure d'un temps de calcul. Un ressort de compression en acier, de module de cisaillement $G = 79\,000$ MPa, doit porter $250$ N avec une raideur de $10$ N/mm à $10\,\%$ près, sans que la contrainte de cisaillement dépasse $600$ MPa. Trois grandeurs se choisissent : le diamètre $d$ du fil, le rapport d'enroulement $C = D/d$, où $D$ est le diamètre moyen du ressort, et le nombre $n$ de spires actives. Plutôt que de tâtonner, on tire 10 000 conceptions au hasard, on calcule leurs contraintes et leurs raideurs sans une seule boucle, on garde les conceptions admissibles et l'on cherche la plus légère. Aucun jeu de données : le tirage se refait à l'identique grâce à sa graine. La figure du chapitre donne la feuille de route.

**Ressort de compression.** Facteur de Wahl : $K_W = \dfrac{4C - 1}{4C - 4} + \dfrac{0{,}615}{C}$. Contrainte : $\tau = K_W\,\dfrac{8\,F\,D}{\pi\,d^3}$.\hspace{1.5em} Raideur : $k = \dfrac{G\,d^4}{8\,D^3\,n}$. Masse : $m = \rho\,\dfrac{\pi d^2}{4}\,\pi D\,(n + 2)$, fil de $n + 2$ spires, $\rho = 7{,}85$ g/cm³, soit $7{,}85 \times 10^{-3}$ g/mm³.

## Source de données

Aucun jeu de données : les valeurs viennent de l'énoncé ou sont engendrées par le code, à graine fixée.

### En-tête de reproductibilité

Versions des bibliothèques employées : les valeurs du livre sont celles de ces versions, avec des graines fixées.

In [1]:
# En-tête de reproductibilité : versions employées par ce notebook.
import platform
from importlib.metadata import version

print("Python", platform.python_version())
print("matplotlib", version("matplotlib"))
print("numpy", version("numpy"))

Python 3.12.0
matplotlib 3.11.1
numpy 2.4.6


### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas les dossiers du dépôt. La cellule suivante ne télécharge rien : elle cherche `donnees/` et `outils/` (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-PY_Ed2026`), ajoute `outils/` au chemin de recherche de Python, et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez les dossiers `donnees` et `outils` à la racine de l'espace de travail, et téléversez-y les fichiers listés, pris dans le dépôt. Ce notebook n'a besoin d'aucun jeu ni d'aucun module : la cellule ne signale rien, mais elle prépare l'import des modules du livre dans vos propres cellules.

In [2]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
import sys
from pathlib import Path

JEUX_LUS = []
MODULES_DU_LIVRE = []                          # fichiers de outils/


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-PY_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees, outils = chercher("donnees"), chercher("outils")
if outils is not None and str(outils) not in sys.path:
    sys.path.insert(0, str(outils))
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
manque += [f"outils/{nom}" for nom in MODULES_DU_LIVRE if outils is None or not (outils / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Préparation

In [3]:
import math
import timeit
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import to_rgb
G_MPa, F_N, K_VISEE, TAU_ADM_MPa, MASSE_VOLUMIQUE = 79000.0, 250.0, 10.0, 600.0, 7.85e-3     # MPa, N, N/mm, MPa, g/mm3

def wahl(rapport_c):
    """Facteur de Wahl d'un ressort : (4 C - 1) / (4 C - 4) + 0,615 / C, avec C = D / d."""
    return (4 * rapport_c - 1) / (4 * rapport_c - 4) + 0.615 / rapport_c

## Étape 1 : tirage des conceptions

Tirage des conceptions.

Les contrôles de la cellule lisent les variables `D_mm`, `d_mm`, `spires` : gardez ces noms.

In [4]:
# Solution :
rng = np.random.default_rng(2026)
n_conceptions = 10000
d_mm = rng.uniform(2.0, 5.0, n_conceptions)                  # diamètre du fil
rapport = rng.uniform(5.0, 12.0, n_conceptions)               # rapport d'enroulement C = D / d
spires = rng.integers(4, 16, n_conceptions)                   # nombre de spires actives, de 4 à 15
D_mm = rapport * d_mm                                         # diamètre moyen d'enroulement
print("conceptions :", d_mm.shape, "; diamètres de", round(float(d_mm.min()), 2), "à", round(float(d_mm.max()), 2), "mm")
# Fin de la solution
assert d_mm.shape == D_mm.shape == spires.shape == (10000,), "dix mille conceptions"
assert 2.0 <= d_mm.min() and d_mm.max() <= 5.0 and spires.min() == 4 and spires.max() == 15
print("Étape 1 : contrôles passés")

conceptions : (10000,) ; diamètres de 2.0 à 5.0 mm
Étape 1 : contrôles passés


## Étape 2 : contraintes et raideurs

Contrainte de Wahl et raideur, sans boucle ; les conceptions admissibles par un masque combiné.

Les contrôles de la cellule lisent les variables `admissible`, `n_admissibles`, `raideur`, `tau_MPa` : gardez ces noms.

In [5]:
# Solution :
tau_MPa = wahl(rapport) * 8 * F_N * D_mm / (math.pi * d_mm**3)
raideur = G_MPa * d_mm**4 / (8 * D_mm**3 * spires)           # N/mm
masse_g = MASSE_VOLUMIQUE * (math.pi * d_mm**2 / 4) * (math.pi * D_mm * (spires + 2))     # fil de n + 2 spires
admissible = (tau_MPa <= TAU_ADM_MPa) & (np.abs(raideur - K_VISEE) <= 0.1 * K_VISEE)
n_admissibles = int(admissible.sum())
print(f"contrainte moyenne {tau_MPa.mean():.1f} MPa, écart-type {tau_MPa.std(ddof=1):.1f} MPa ; sous la limite : {int((tau_MPa <= TAU_ADM_MPa).sum())}")
print(f"conceptions admissibles : {n_admissibles} sur {n_conceptions}")
# Fin de la solution
assert abs(wahl(8.0) - 1.1840) < 0.0001, "facteur de Wahl pour C = 8 : 1,1840"
assert (
    tau_MPa.shape == (10000,) and abs(tau_MPa.mean() - 625.7) < 0.1
), "contrainte moyenne attendue : 625,7 MPa"
assert n_admissibles == 522, "conceptions admissibles attendues : 522"
assert np.all(tau_MPa[admissible] <= TAU_ADM_MPa), "contrainte admissible"
assert np.all(np.abs(raideur[admissible] - K_VISEE) <= 1.0), "raideur à 10 % près"
print("Étape 2 : contrôles passés")

contrainte moyenne 625.7 MPa, écart-type 368.8 MPa ; sous la limite : 5928
conceptions admissibles : 522 sur 10000
Étape 2 : contrôles passés


## Étape 3 : conception la plus légère

Conception la plus légère parmi les admissibles.

Les contrôles de la cellule lisent la variable `i_mini` : gardez ce nom.

In [6]:
# Solution :
masse_admissible = np.where(admissible, masse_g, np.inf)
i_mini = int(np.argmin(masse_admissible))
print(f"fil {d_mm[i_mini]:.2f} mm, diamètre moyen {D_mm[i_mini]:.2f} mm, {spires[i_mini]} spires ; {masse_g[i_mini]:.1f} g ;"
      f" contrainte {tau_MPa[i_mini]:.1f} MPa ; raideur {raideur[i_mini]:.2f} N/mm")
# Fin de la solution
assert (
    abs(d_mm[i_mini] - 2.91) < 0.005 and abs(D_mm[i_mini] - 18.75) < 0.005 and spires[i_mini] == 10
), "conception attendue"
assert abs(masse_g[i_mini] - 36.8) < 0.05 and masse_g[i_mini] == masse_g[admissible].min()
assert tau_MPa[i_mini] <= TAU_ADM_MPa and abs(raideur[i_mini] - K_VISEE) <= 1.0
print("Étape 3 : contrôles passés")

fil 2.91 mm, diamètre moyen 18.75 mm, 10 spires ; 36.8 g ; contrainte 598.6 MPa ; raideur 10.72 N/mm
Étape 3 : contrôles passés


## Étape 4 : lisibilité en gris

Deux histogrammes de la contrainte, lisibles en gris : toutes les conceptions et les admissibles.

Les contrôles de la cellule lisent les variables `bas`, `ecart_gris`, `haut`, `styles` : gardez ces noms.

In [7]:
# Solution :
fig, (haut, bas) = plt.subplots(2, 1, figsize=(3.2, 3.4), sharex=True)
styles = [("toutes", "tab:blue", "solid", ""), ("admissibles", "tab:orange", "dashed", "//")]
haut.hist(tau_MPa[tau_MPa < 1500.0], bins=40, color=styles[0][1], label=styles[0][0])
bas.hist(tau_MPa[admissible], bins=20, color=styles[1][1], hatch=styles[1][3], label=styles[1][0])
haut.axvline(TAU_ADM_MPa, color="black", linestyle="dashed")
bas.set_xlabel("Contrainte de cisaillement (MPa)")
haut.set_ylabel("Effectif")
bas.set_ylabel("Effectif")
haut.legend(frameon=False)
bas.legend(frameon=False)
plt.close(fig)
gris = []

for nom, couleur, trait, hachure in styles:
    r, v, b = to_rgb(couleur)
    gris.append(round(255 * (0.299 * r + 0.587 * v + 0.114 * b)))        # luminance perçue, 0 à 255
    print(f"{nom:>12} : couleur {couleur:<10} gris {gris[-1]:>3}, hachure {hachure!r}")

ecart_gris = abs(gris[0] - gris[1])
print("écart de gris entre les deux séries :", ecart_gris, "niveaux sur 255 ; hachures distinctes :", styles[0][3] != styles[1][3])
# Fin de la solution
assert ecart_gris >= 25 and styles[0][3] != styles[1][3], "séparées par le gris ou par la hachure"
assert (
    (haut.get_xlabel(), bas.get_xlabel()) == ("", "Contrainte de cisaillement (MPa)")
), "un seul axe horizontal étiqueté"
print("Étape 4 : contrôles passés")

      toutes : couleur tab:blue   gris 100, hachure ''
 admissibles : couleur tab:orange gris 152, hachure '//'
écart de gris entre les deux séries : 52 niveaux sur 255 ; hachures distinctes : True
Étape 4 : contrôles passés


## Étape 5 : types et temps de calcul

Le type float32 change-t-il la décision, et que gagne-t-on en temps et en mémoire ?

Les contrôles de la cellule lisent les variables `decisions_changees`, `duree_boucle`, `duree_tableau`, `tau_boucle`, `tau_simple` : gardez ces noms.

In [8]:
# Solution :
tau_simple = (wahl(rapport.astype(np.float32)) * 8 * np.float32(F_N) * D_mm.astype(np.float32)
              / (np.float32(math.pi) * d_mm.astype(np.float32)**3))
decisions_changees = int(np.sum((tau_simple <= TAU_ADM_MPa) != (tau_MPa <= TAU_ADM_MPa)))
print("types :", tau_simple.dtype, "et", tau_MPa.dtype, "; octets :", tau_simple.nbytes, "et", tau_MPa.nbytes)
print("conceptions dont la décision change en float32 :", decisions_changees)
# Fin de la solution
assert (
    tau_simple.dtype == np.float32 and tau_simple.nbytes * 2 == tau_MPa.nbytes
), "moitié moins de mémoire"
assert (
    decisions_changees <= 2
), "float32 ne change la décision que pour des conceptions au ras de la limite"
# Solution :
def tau_boucle(c, d, F):
    """Contrainte de Wahl de chaque conception, par une boucle sur les indices."""
    sortie = np.zeros(len(c))
    for i in range(len(c)):
        sortie[i] = ((4 * c[i] - 1) / (4 * c[i] - 4) + 0.615 / c[i]) * 8 * F * c[i] * d[i] / (math.pi * d[i]**3)
    return sortie
# Fin de la solution
assert (
    np.allclose(tau_boucle(rapport, d_mm, F_N), tau_MPa)
), "la boucle et le tableau donnent les mêmes contraintes"
# Solution :
duree_boucle = min(timeit.repeat(lambda: tau_boucle(rapport, d_mm, F_N), number=1, repeat=3))
duree_tableau = min(timeit.repeat(lambda: wahl(rapport) * 8 * F_N * (rapport * d_mm) / (math.pi * d_mm**3), number=20, repeat=3)) / 20
print("le calcul par tableau est au moins 10 fois plus rapide que la boucle :", duree_boucle / duree_tableau > 10)
# Fin de la solution
assert duree_boucle / duree_tableau > 10, "le tableau calcule au moins 10 fois plus vite"
# Solution :
print("étude II vérifiée")
# Fin de la solution
print("Étape 5 : contrôles passés")

types : float32 et float64 ; octets : 40000 et 80000
conceptions dont la décision change en float32 : 0
le calcul par tableau est au moins 10 fois plus rapide que la boucle : True
étude II vérifiée
Étape 5 : contrôles passés
